# 🧹 Phase 4: Preprocessing — LPDP

---

## 🎯 Tujuan Notebook
Membersihkan dan menormalisasi **1.038 artikel LPDP berkonten** melalui **10 langkah preprocessing** terstandar untuk menghasilkan kolom `text_clean` yang siap digunakan oleh Phase 5 (Feature Extraction), Phase 6 (NER), dan Phase 8 (Sentiment Lexicon).

### Alur 10 Langkah:
| Langkah | Deskripsi |
|---------|----------|
| 1️⃣ Case Folding | Lowercase semua teks |
| 2️⃣ Remove URL | Hapus `http://`, `https://`, `www.` |
| 3️⃣ Remove Mention & Hashtag | Hapus `@user` dan `#topic` |
| 4️⃣ Remove Digit & Punctuation | Hapus angka + tanda baca |
| 5️⃣ Slang Normalization | Normalkan bahasa informal → formal (kamus custom CSV) |
| 6️⃣ Tokenization | Pecah teks → list token (`nltk.word_tokenize`) |
| 7️⃣ Stopword Removal | Hapus kata umum (NLTK Indonesian 757 kata + augmentasi) |
| 8️⃣ Stemming | Reduksi morfologi → kata dasar (Sastrawi) |
| 9️⃣ Rare Word Removal | Hapus kata frekuensi < 2 di seluruh korpus |
| 🔟 Join Tokens | Gabung token → string akhir `text_clean` |

---

**Kelompok 5:** Amel, Celine, Iqbal, Nida, Salwa

**PIC Phase 4:** Iqbal

**Input:** `output_bertopic/bertopic_4_topik_final.xlsx` — 1.038 artikel + kolom `topic_label` (Phase 3 output)

**Output:** `dataset_lpdp_preprocessed.csv` — kolom `text_clean` + semua kolom sebelumnya

## 📦 Install Dependencies

Library yang dibutuhkan untuk Phase 4:
- **PySastrawi**: Stemmer morfologi Bahasa Indonesia (support imbuhan me-, di-, ke-an, pe-an, dll.)
- **nltk**: Tokenisasi + stopword list Bahasa Indonesia (757 kata)
- **pandas** / **openpyxl**: Baca file Excel (Phase 3 output) + simpan CSV

> ⚠️ Gunakan kernel **Python (.venv312) - gensim** untuk konsistensi environment dengan Phase 3.

In [5]:
import sys
import subprocess

def install_packages(packages: list[str]) -> None:
    # Pastikan pip tersedia di kernel aktif
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

install_packages(["Sastrawi", "nltk", "pandas", "openpyxl", "numpy"])

print("Install selesai. Jika ini pertama kali install package di kernel ini, restart kernel lalu jalankan ulang dari Cell 3.")

Install selesai. Jika ini pertama kali install package di kernel ini, restart kernel lalu jalankan ulang dari Cell 3.


## 📦 Import Libraries

Import semua library + download NLTK data (stopwords, punkt) yang dibutuhkan.

In [7]:
import re
import string
import os
import sys
from collections import Counter

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory  # type: ignore[reportMissingImports]

# Download NLTK data (hanya perlu sekali)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# ── Verifikasi ──
print(f"python     : {sys.executable}")
print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"nltk       : {nltk.__version__}")
import Sastrawi  # type: ignore[reportMissingImports]
print(f"PySastrawi : {getattr(Sastrawi, '__version__', 'installed')}")
print()
print("✅ Semua library berhasil diimport!")

python     : c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv312\Scripts\python.exe
pandas     : 3.0.2
numpy      : 2.4.4
nltk       : 3.9.4
PySastrawi : installed

✅ Semua library berhasil diimport!


## ⚙️ Konfigurasi

Semua path input/output dan parameter threshold dipusatkan di sini agar mudah diubah.

In [8]:
# ── Path ──────────────────────────────────────────────────────────
BASE_DIR      = os.path.dirname(os.path.abspath('__file__'))
INPUT_FILE    = os.path.join(BASE_DIR, 'output_bertopic', 'bertopic_4_topik_final.xlsx')
OUTPUT_FILE   = os.path.join(BASE_DIR, 'dataset_lpdp_preprocessed.csv')
SLANG_FILE    = os.path.join(BASE_DIR, 'slang_id.csv')   # kamus slang (dibuat di Step 5)

# ── Parameter ─────────────────────────────────────────────────────
RARE_FREQ_THRESHOLD = 2      # Step 9: hapus kata dengan frekuensi < threshold
TEXT_COLUMN         = 'Content'  # kolom teks mentah yang akan diproses
RANDOM_SEED         = 42

np.random.seed(RANDOM_SEED)

print(f"INPUT  : {INPUT_FILE}")
print(f"OUTPUT : {OUTPUT_FILE}")
print(f"Rare word threshold : freq < {RARE_FREQ_THRESHOLD}")

INPUT  : c:\Users\mikba\Downloads\Documents\PBA\PBA\Project A\output_bertopic\bertopic_4_topik_final.xlsx
OUTPUT : c:\Users\mikba\Downloads\Documents\PBA\PBA\Project A\dataset_lpdp_preprocessed.csv
Rare word threshold : freq < 2


## 📂 Load Data (Phase 3 Output)

Input: `bertopic_4_topik_final.xlsx` — hasil Phase 3 yang sudah memiliki kolom `topic_label` (4 topik: Beasiswa, Alumni, Kebijakan, Akademik).

In [12]:
df = pd.read_excel(INPUT_FILE)

# Auto-detect kolom topik agar tidak hardcode satu nama
topic_candidates = [
    'topic_label', 'Topic_Label', 'topic', 'Topic', 'Topik', 'Label_Topik',
    'label_name', 'label_4'
 ]
TOPIC_COLUMN = next((c for c in topic_candidates if c in df.columns), None)

print(f"Shape     : {df.shape}")
print(f"Kolom     : {list(df.columns)}")
print(f"\nDistribusi Label Sentimen:")
print(df['Sentiment'].value_counts())
if TOPIC_COLUMN:
    print(f"\nDistribusi Topic Label ({TOPIC_COLUMN}):")
    print(df[TOPIC_COLUMN].value_counts())
else:
    print("\n⚠️ Kolom topik tidak ditemukan. Proses lanjut tanpa analisis per topik.")
print(f"\nContoh teks mentah (baris 0):")
print(df[TEXT_COLUMN].iloc[0][:300])

Shape     : (1038, 13)
Kolom     : ['doc_id', 'Title', 'Release Date', 'URL', 'Publisher', 'PiC', 'Valid?', 'Sentiment', 'Notes', 'Actual_URL', 'Content', 'label_4', 'label_name']

Distribusi Label Sentimen:
Sentiment
Positive    385
Neutral     342
Negative    311
Name: count, dtype: int64

Distribusi Topic Label (label_name):
label_name
Kebijakan & Prioritas Program    553
Kewajiban & Sanksi Penerima      147
Pendaftaran & Seleksi LPDP       140
Kontroversi Penerima Beasiswa     97
Name: count, dtype: int64

Contoh teks mentah (baris 0):
Cegah kolusi-nepotisme, program LPDP dinilai perlu perketat seleksi

Jakarta (ANTARA) - Sekretaris Komisi E DPRD DKI Jakarta Justin Adrian Untayana menilai seleksi penerima program beasiswa Lembaga Pengelola Dana Pendidikan (LPDP) versi Jakarta perlu diperketat untuk mencegah praktik kolusi dan nepo


---

## 1️⃣ Step 1 — Case Folding

Ubah semua teks ke **huruf kecil** (lowercase). Ini memastikan `"LPDP"` dan `"lpdp"` diperlakukan sebagai token yang sama.

```python
"Alumni LPDP Mendapat Beasiswa" → "alumni lpdp mendapat beasiswa"
```

In [13]:
def step1_case_folding(text: str) -> str:
    """Ubah semua karakter ke lowercase."""
    if pd.isna(text):
        return ''
    return str(text).lower()

# ── Test ──
sample = "Alumni LPDP Mendapat Beasiswa S2 di UNIVERSITAS INDONESIA!"
print(f"Before : {sample}")
print(f"After  : {step1_case_folding(sample)}")

Before : Alumni LPDP Mendapat Beasiswa S2 di UNIVERSITAS INDONESIA!
After  : alumni lpdp mendapat beasiswa s2 di universitas indonesia!


---

## 2️⃣ Step 2 — Remove URL

Hapus semua URL dalam teks (`http://`, `https://`, `www.`, dll.). URL tidak berkontribusi pada analisis sentimen/topik.

```python
"kunjungi https://lpdp.kemenkeu.go.id untuk info" → "kunjungi  untuk info"
```

In [14]:
URL_PATTERN = re.compile(
    r'(?:https?://|www\.)\S+|'
    r'\S+\.(?:com|id|net|org|gov|co\.id|go\.id)(?:/\S*)?',
    re.IGNORECASE
)

def step2_remove_url(text: str) -> str:
    """Hapus semua URL dari teks."""
    return URL_PATTERN.sub(' ', text).strip()

# ── Test ──
sample = "kunjungi https://lpdp.kemenkeu.go.id atau www.beasiswa.id untuk info lengkap."
print(f"Before : {sample}")
print(f"After  : {step2_remove_url(sample)}")

Before : kunjungi https://lpdp.kemenkeu.go.id atau www.beasiswa.id untuk info lengkap.
After  : kunjungi   atau   untuk info lengkap.


---

## 3️⃣ Step 3 — Remove Mention & Hashtag

Hapus mention (`@user`) dan hashtag (`#topik`) yang umum dalam konten kutipan media sosial dalam artikel berita.

```python
"@kompas #LPDP2024 terimakasih" → " terimakasih"
```

In [15]:
MENTION_HASHTAG_PATTERN = re.compile(r'[@#]\w+')

def step3_remove_mention_hashtag(text: str) -> str:
    """Hapus @mention dan #hashtag."""
    return MENTION_HASHTAG_PATTERN.sub(' ', text).strip()

# ── Test ──
sample = "@kompas_id bilang #LPDP2024 buka pendaftaran gelombang 2"
print(f"Before : {sample}")
print(f"After  : {step3_remove_mention_hashtag(sample)}")

Before : @kompas_id bilang #LPDP2024 buka pendaftaran gelombang 2
After  : bilang   buka pendaftaran gelombang 2


---

## 4️⃣ Step 4 — Remove Digit & Punctuation

Hapus semua angka dan tanda baca. Angka (tahun, nominal) dan tanda baca tidak relevan untuk analisis semantik berbasis token.

```python
"tahun 2024, nilai 95.5!" → "tahun  nilai  "
```

In [16]:
DIGIT_PUNCT_PATTERN = re.compile(r'[0-9' + re.escape(string.punctuation) + r']+')

def step4_remove_digit_punct(text: str) -> str:
    """Hapus semua angka dan tanda baca."""
    cleaned = DIGIT_PUNCT_PATTERN.sub(' ', text)
    # Hapus spasi berlebih
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

# ── Test ──
sample = "anggaran lpdp tahun 2024 sebesar rp 20.000.000.000, naik 15% dari 2023!"
print(f"Before : {sample}")
print(f"After  : {step4_remove_digit_punct(sample)}")

Before : anggaran lpdp tahun 2024 sebesar rp 20.000.000.000, naik 15% dari 2023!
After  : anggaran lpdp tahun sebesar rp naik dari


---

## 5️⃣ Step 5 — Slang Normalization

Normalisasi bahasa informal → formal menggunakan **kamus slang kustom**. Artikel berita sering mengutip komentar netizen/media sosial yang mengandung bahasa tidak baku.

Kamus slang disimpan sebagai `slang_id.csv` (dua kolom: `slang`, `formal`) untuk memudahkan maintenance dan penambahan entri baru.

In [17]:
# ── Kamus Slang Indonesia ──────────────────────────────────────────
SLANG_RAW = {
    # Negasi
    "gak": "tidak", "ga": "tidak", "ngga": "tidak", "nggak": "tidak",
    "enggak": "tidak", "tdk": "tidak", "tak": "tidak",
    # Intensifier
    "bgt": "sangat", "banget": "sangat", "amat": "sangat",
    # Preposisi / Konjungsi singkat
    "yg": "yang", "dg": "dengan", "dgn": "dengan", "dr": "dari",
    "utk": "untuk", "buat": "untuk", "krn": "karena", "karna": "karena",
    "krena": "karena", "sbg": "sebagai", "sbgai": "sebagai",
    "pd": "pada", "tp": "tetapi", "tapi": "tetapi", "ttg": "tentang",
    "thd": "terhadap", "thdp": "terhadap", "kpd": "kepada",
    "dlm": "dalam", "tsb": "tersebut", "spy": "supaya",
    # Waktu
    "sdh": "sudah", "udah": "sudah", "udh": "sudah",
    "blm": "belum", "belom": "belum",
    "skrg": "sekarang", "skrang": "sekarang",
    "kmrn": "kemarin", "bsk": "besok", "ntar": "nanti",
    "thn": "tahun", "bln": "bulan", "mgg": "minggu",
    "tgl": "tanggal", "trs": "terus", "trus": "terus",
    # Modalitas
    "bs": "bisa", "hrs": "harus", "msh": "masih", "masi": "masih",
    "jd": "jadi", "sm": "sama", "pst": "pasti", "psti": "pasti",
    # Kondisi / Perbandingan
    "klo": "kalau", "kalo": "kalau", "kaya": "seperti",
    "kayak": "seperti", "kayaknya": "sepertinya",
    "gimana": "bagaimana", "gmn": "bagaimana", "gmna": "bagaimana",
    "kenapa": "mengapa", "knp": "mengapa",
    # Verba informal
    "bikin": "membuat", "ngerti": "mengerti", "pake": "menggunakan",
    "nyoba": "mencoba", "ngomong": "berbicara", "bilang": "mengatakan",
    "liat": "melihat", "dapet": "mendapat", "nyari": "mencari",
    "kerjain": "mengerjakan", "mikir": "berpikir", "nunggu": "menunggu",
    "balik": "kembali",
    # Nomina / Pronomina
    "org": "orang", "orng": "orang", "punya": "memiliki",
    "nih": "ini", "tuh": "itu",
    # Partikel (dihapus)
    "sih": "", "deh": "", "dong": "", "lho": "",
    "lah": "", "mah": "", "kan": "",
    # Informal lain
    "aja": "saja", "doang": "saja", "cuma": "hanya", "cman": "hanya",
    "emang": "memang", "emg": "memang",
    "beneran": "benar", "bener": "benar", "bnr": "benar",
    "susah": "sulit", "gampang": "mudah",
    "jt": "juta", "rb": "ribu",
    "dll": "dan lain lain", "dsb": "dan sebagainya", "dst": "dan seterusnya",
    "yuk": "ayo", "makanya": "maka", "makasih": "terima kasih",
    "dikit": "sedikit", "sdkt": "sedikit", "bnyk": "banyak",
    "ttd": "tanda tangan",
}

# Simpan ke CSV untuk referensi
slang_df = pd.DataFrame(list(SLANG_RAW.items()), columns=['slang', 'formal'])
slang_df.to_csv(SLANG_FILE, index=False)

print(f"Kamus slang: {len(SLANG_RAW)} entri → disimpan ke {SLANG_FILE}")
print(slang_df.head(10).to_string(index=False))

Kamus slang: 113 entri → disimpan ke c:\Users\mikba\Downloads\Documents\PBA\PBA\Project A\slang_id.csv
 slang formal
   gak  tidak
    ga  tidak
  ngga  tidak
 nggak  tidak
enggak  tidak
   tdk  tidak
   tak  tidak
   bgt sangat
banget sangat
  amat sangat


In [18]:
def step5_normalize_slang(text: str, slang_dict: dict) -> str:
    """Ganti kata slang dengan bentuk formalnya (word-level replacement)."""
    tokens = text.split()
    normalized = [slang_dict.get(tok, tok) for tok in tokens]
    # Hapus token kosong (partikel yang dihapus)
    normalized = [t for t in normalized if t]
    return ' '.join(normalized)

# ── Test ──
sample = "gak ada yang tau kenapa alumni lpdp bgt susah dapet kerja kan"
print(f"Before : {sample}")
print(f"After  : {step5_normalize_slang(sample, SLANG_RAW)}")

Before : gak ada yang tau kenapa alumni lpdp bgt susah dapet kerja kan
After  : tidak ada yang tau mengapa alumni lpdp sangat sulit mendapat kerja


---

## 6️⃣ Step 6 — Tokenization

Pecah string teks menjadi **list token** menggunakan `nltk.word_tokenize` dengan bahasa Indonesia.

```python
"alumni lpdp sukses meraih beasiswa" → ["alumni", "lpdp", "sukses", "meraih", "beasiswa"]
```

In [ ]:
def step6_tokenize(text: str) -> list:
    """Tokenisasi teks → list token. Fallback jika resource punkt_tab ID tidak tersedia."""
    try:
        return word_tokenize(text, language='indonesian')
    except LookupError:
        # Fallback aman: tidak butuh punkt_tab/indonesian
        return nltk.tokenize.wordpunct_tokenize(text)

# ── Test ──
sample = "alumni lpdp sukses meraih beasiswa untuk studi lanjut di luar negeri"
tokens = step6_tokenize(sample)
print(f"Input  : {sample}")
print(f"Tokens : {tokens}")
print(f"Jumlah : {len(tokens)} token")

LookupError: 
**********************************************************************
  Resource entry 'tokenizers/punkt_tab/indonesian/' not found in installed package 'punkt_tab'.
  The package appears to be installed, but the requested file is missing.

  If you believe the package is corrupted or out of date, you can try re-downloading it with the NLTK Downloader:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/indonesian/'

  Package was found in:
    - 'C:\\Users\\mikba\\AppData\\Roaming\\nltk_data\\tokenizers\\punkt_tab'
  Searched in:
    - 'C:\\Users\\mikba/nltk_data'
    - 'c:\\Users\\mikba\\Downloads\\Documents\\PBA\\PBA\\.venv312\\nltk_data'
    - 'c:\\Users\\mikba\\Downloads\\Documents\\PBA\\PBA\\.venv312\\share\\nltk_data'
    - 'c:\\Users\\mikba\\Downloads\\Documents\\PBA\\PBA\\.venv312\\lib\\nltk_data'
    - 'C:\\Users\\mikba\\AppData\\Roaming\\nltk_data'
    - 'C:\

---

## 7️⃣ Step 7 — Stopword Removal

Hapus kata-kata umum yang tidak berkontribusi pada makna semantik. Menggunakan **NLTK Indonesian stopwords (757 kata)** yang diaugmentasi dengan stopword domain-spesifik LPDP.

Kata seperti `"yang"`, `"dan"`, `"di"`, `"ini"`, `"itu"` dihapus karena muncul di hampir semua dokumen.

In [ ]:
# ── Stopword List ──────────────────────────────────────────────────
ID_STOPWORDS = set(stopwords.words('indonesian'))

# Augmentasi stopword domain-spesifik
EXTRA_STOPWORDS = {
    # Kata umum artikel berita
    'kata', 'ujar', 'tutur', 'ungkap', 'sebut', 'jelaskan', 'nyatakan',
    'sampaikan', 'tambah', 'lanjut', 'terang', 'ucap', 'tanya', 'jawab',
    # Kata transisi umum
    'saat', 'ketika', 'selain', 'namun', 'namanya', 'hal', 'bisa',
    'dapat', 'akan', 'telah', 'sedang', 'masih', 'sudah', 'belum',
    # Angka/satuan sisa (setelah step 4 ada yang lolos)
    'rp', 'rb', 'jt', 'm', 'km',
    # Singkatan umum tidak bermakna
    'dll', 'dsb', 'dst', 'svp', 'rs',
}

ALL_STOPWORDS = ID_STOPWORDS | EXTRA_STOPWORDS

print(f"NLTK Indonesian stopwords : {len(ID_STOPWORDS)} kata")
print(f"Extra domain stopwords    : {len(EXTRA_STOPWORDS)} kata")
print(f"Total stopwords           : {len(ALL_STOPWORDS)} kata")

def step7_remove_stopwords(tokens: list, stopword_set: set) -> list:
    """Hapus token yang termasuk stopword."""
    return [tok for tok in tokens if tok not in stopword_set]

# ── Test ──
sample_tokens = ['alumni', 'yang', 'telah', 'mendapat', 'beasiswa', 'lpdp', 'di', 'luar', 'negeri']
print(f"\nBefore : {sample_tokens}")
print(f"After  : {step7_remove_stopwords(sample_tokens, ALL_STOPWORDS)}")

---

## 8️⃣ Step 8 — Stemming (Sastrawi)

Reduksi token ke **kata dasar** menggunakan **Sastrawi** (algoritma Confix-Stripping). Sastrawi memahami morfologi Bahasa Indonesia secara menyeluruh:

| Token | Kata Dasar |
|-------|------------|
| `pendidikan` | `didik` |
| `penerima` | `terima` |
| `memberikan` | `beri` |
| `keberhasilan` | `hasil` |
| `penyelenggaraan` | `selenggara` |

> ⏱️ **Perhatian**: Stemming Sastrawi pada 1.038 artikel (~200–1000 kata/artikel) membutuhkan **3–8 menit**. Progress bar akan ditampilkan.

In [ ]:
# Inisialisasi stemmer (lakukan sekali saja — butuh ~5 detik load)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

print("✅ Sastrawi stemmer berhasil diinisialisasi.")

# ── Test beberapa kata ──
test_words = ['pendidikan', 'penerima', 'memberikan', 'keberhasilan',
              'penyelenggaraan', 'pembiayaan', 'permasalahan', 'ketidaksesuaian']
print("\nContoh stemming:")
for w in test_words:
    print(f"  {w:25s} → {stemmer.stem(w)}")

In [ ]:
def step8_stem(tokens: list, stemmer_obj) -> list:
    """Stem setiap token ke kata dasar menggunakan Sastrawi."""
    return [stemmer_obj.stem(tok) for tok in tokens]

# ── Test ──
sample_tokens = ['penerima', 'beasiswa', 'lpdp', 'berhasil', 'menyelesaikan', 'pendidikan']
print(f"Before : {sample_tokens}")
print(f"After  : {step8_stem(sample_tokens, stemmer)}")

---

## 9️⃣ Step 9 — Rare Word Removal

Hapus kata yang hanya muncul **kurang dari 2 kali** di seluruh korpus. Kata yang sangat jarang (hapax legomena) biasanya merupakan noise (typo, named entity spesifik, kata asing tidak dikenal) dan tidak berkontribusi pada model ML.

**Threshold**: `frekuensi < {RARE_FREQ_THRESHOLD}` (default: 2)

In [ ]:
def build_vocab_counter(token_lists: list) -> Counter:
    """Bangun Counter frekuensi seluruh token di korpus."""
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)
    return counter

def step9_remove_rare(tokens: list, vocab_counter: Counter, threshold: int = 2) -> list:
    """Hapus token dengan frekuensi korpus < threshold."""
    return [tok for tok in tokens if vocab_counter[tok] >= threshold]

# Vocab counter dibangun SETELAH pipeline dijalankan pada seluruh dataset
# (lihat sel Pipeline di bawah)
print("✅ Fungsi step9_remove_rare siap. Vocab counter dibangun setelah pipeline run awal.")

---

## 🔟 Step 10 — Join Tokens

Gabungkan kembali list token menjadi **string tunggal** `text_clean` yang siap digunakan sebagai input untuk Phase 5 (TF-IDF/BoW) dan Phase 8 (sentiment lexicon).

```python
["alumni", "lpdp", "berhasil", "raih", "beasiswa"] → "alumni lpdp berhasil raih beasiswa"
```

In [ ]:
def step10_join(tokens: list) -> str:
    """Gabung token → string teks bersih."""
    return ' '.join(tokens)

# ── Test ──
sample_tokens = ['alumni', 'lpdp', 'berhasil', 'raih', 'beasiswa']
print(f"Tokens : {sample_tokens}")
print(f"Joined : {step10_join(sample_tokens)}")

---

## 🚀 Full Pipeline — Apply ke Seluruh Dataset

Jalankan 10 langkah preprocessing secara berurutan pada seluruh kolom `Content` di dataset.

**Strategi dua-pass**:
1. **Pass 1** — Jalankan Step 1–8 pada semua baris → hasilkan `tokens_stemmed`
2. **Build Vocab Counter** — Hitung frekuensi tiap kata di seluruh korpus
3. **Pass 2** — Terapkan Step 9 (rare word removal) + Step 10 (join) → `text_clean`

In [ ]:
import time

def preprocess_pass1(text: str, slang_dict: dict, stopword_set: set, stemmer_obj) -> list:
    """Jalankan Step 1–8 pada satu teks, return list token stemmed."""
    text = step1_case_folding(text)
    text = step2_remove_url(text)
    text = step3_remove_mention_hashtag(text)
    text = step4_remove_digit_punct(text)
    text = step5_normalize_slang(text, slang_dict)
    tokens = step6_tokenize(text)
    tokens = step7_remove_stopwords(tokens, stopword_set)
    tokens = step8_stem(tokens, stemmer_obj)
    return tokens

# ── Pass 1: Step 1–8 ──────────────────────────────────────────────
print("[Pass 1] Menjalankan Step 1–8 pada 1.038 artikel...")
print("⏱️  Estimasi waktu: 3–8 menit (Sastrawi stemming per token)")
t0 = time.time()

all_tokens_stemmed = []
total = len(df)
for i, text in enumerate(df[TEXT_COLUMN]):
    tokens = preprocess_pass1(text, SLANG_RAW, ALL_STOPWORDS, stemmer)
    all_tokens_stemmed.append(tokens)
    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta = (elapsed / (i + 1)) * (total - i - 1)
        print(f"  Progress: {i+1}/{total} ({(i+1)/total*100:.1f}%) | Elapsed: {elapsed:.0f}s | ETA: {eta:.0f}s")

elapsed = time.time() - t0
print(f"\n✅ Pass 1 selesai dalam {elapsed:.1f} detik ({elapsed/60:.1f} menit)")
print(f"   Rata-rata token per artikel (setelah Step 8): {np.mean([len(t) for t in all_tokens_stemmed]):.1f}")

In [ ]:
# ── Build Vocab Counter ───────────────────────────────────────────
vocab_counter = build_vocab_counter(all_tokens_stemmed)

total_vocab  = len(vocab_counter)
rare_count   = sum(1 for v in vocab_counter.values() if v < RARE_FREQ_THRESHOLD)
keep_count   = total_vocab - rare_count

print(f"Total kosakata (setelah Step 8)  : {total_vocab:,} kata")
print(f"Kata rare (freq < {RARE_FREQ_THRESHOLD})            : {rare_count:,} ({rare_count/total_vocab*100:.1f}%)")
print(f"Kata dipertahankan               : {keep_count:,} ({keep_count/total_vocab*100:.1f}%)")
print(f"\nTop-20 kata paling sering:")
for word, freq in vocab_counter.most_common(20):
    print(f"  {word:20s} : {freq}")

In [ ]:
# ── Pass 2: Step 9 + 10 ───────────────────────────────────────────
print("[Pass 2] Menerapkan Step 9 (rare removal) + Step 10 (join)...")

text_clean_list = []
for tokens in all_tokens_stemmed:
    tokens = step9_remove_rare(tokens, vocab_counter, RARE_FREQ_THRESHOLD)
    clean  = step10_join(tokens)
    text_clean_list.append(clean)

df['text_clean'] = text_clean_list

print(f"✅ Pass 2 selesai. Kolom 'text_clean' berhasil ditambahkan.")
print(f"\nContoh hasil (baris 0):")
print(f"  Original ({len(str(df[TEXT_COLUMN].iloc[0]))} char): {str(df[TEXT_COLUMN].iloc[0])[:200]}...")
print(f"  Cleaned  ({len(df['text_clean'].iloc[0])} char): {df['text_clean'].iloc[0][:200]}")

---

## ✅ Validasi Output

Periksa kualitas `text_clean` sebelum disimpan: cek missing values, panjang token, dan distribusi per sentimen/topik.

In [ ]:
# ── 1. Missing / Empty ────────────────────────────────────────────
empty_mask   = df['text_clean'].str.strip().eq('')
na_mask      = df['text_clean'].isna()
n_empty      = empty_mask.sum()
n_na         = na_mask.sum()

print("=" * 55)
print(" VALIDASI OUTPUT text_clean")
print("=" * 55)
print(f"Total artikel    : {len(df)}")
print(f"NaN              : {n_na}")
print(f"Empty string     : {n_empty}")
print(f"Valid            : {len(df) - n_empty - n_na}")

# ── 2. Statistik panjang token ────────────────────────────────────
df['token_count'] = df['text_clean'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

print(f"\nStatistik panjang text_clean (dalam token):")
print(df['token_count'].describe().to_string())

# ── 3. Distribusi per Sentimen ────────────────────────────────────
print(f"\nRata-rata token per Sentimen:")
print(df.groupby('Sentiment')['token_count'].agg(['mean', 'median', 'min', 'max']).to_string())

# ── 4. Distribusi per Topic ───────────────────────────────────────
if 'TOPIC_COLUMN' in globals() and TOPIC_COLUMN:
    print(f"\nRata-rata token per Topic Label ({TOPIC_COLUMN}):")
    print(df.groupby(TOPIC_COLUMN)['token_count'].agg(['mean', 'median']).to_string())
else:
    print("\n⚠️ Skip distribusi per topik (kolom topik tidak tersedia).")

# ── 5. Cek artikel terpendek ──────────────────────────────────────
short_cols = ['Sentiment', 'token_count', 'text_clean']
if 'TOPIC_COLUMN' in globals() and TOPIC_COLUMN:
    short_cols.insert(1, TOPIC_COLUMN)
print(f"\nTop-5 artikel terpendek (token_count):")
print(df.nsmallest(5, 'token_count')[short_cols].to_string())

In [ ]:
# ── 6. Perbandingan teks mentah vs bersih (3 contoh acak) ─────────
import random
random.seed(RANDOM_SEED)
sample_idx = random.sample(range(len(df)), 3)

print("=" * 60)
print(" CONTOH PREPROCESSING (sebelum vs sesudah)")
print("=" * 60)
for i, idx in enumerate(sample_idx, 1):
    row = df.iloc[idx]
    if 'TOPIC_COLUMN' in globals() and TOPIC_COLUMN:
        print(f"\n[{i}] Sentimen: {row['Sentiment']} | Topik: {row[TOPIC_COLUMN]}")
    else:
        print(f"\n[{i}] Sentimen: {row['Sentiment']}")
    print(f"  BEFORE ({len(str(row[TEXT_COLUMN]))} char): {str(row[TEXT_COLUMN])[:200]}...")
    print(f"  AFTER  ({row['token_count']} token): {row['text_clean'][:200]}")

---

## 💾 Simpan Output

Simpan DataFrame dengan kolom `text_clean` ke `dataset_lpdp_preprocessed.csv`. File ini menjadi **input utama** untuk:
- **Phase 5** — Feature extraction (TF-IDF, BoW, IndoBERT embeddings)
- **Phase 8** — Analisis sentimen berbasis leksikon InSet

In [ ]:
# Kolom yang disimpan (hapus kolom helper)
SAVE_COLUMNS = [
    col for col in df.columns
    if col != 'token_count'  # kolom helper — tidak perlu disimpan permanen
]

df[SAVE_COLUMNS].to_csv(OUTPUT_FILE, index=False)

file_size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f"✅ File disimpan ke : {OUTPUT_FILE}")
print(f"   Ukuran file      : {file_size_kb:.1f} KB")
print(f"   Jumlah baris     : {len(df)}")
print(f"   Jumlah kolom     : {len(SAVE_COLUMNS)} → {SAVE_COLUMNS}")

# Verifikasi reload
df_verify = pd.read_csv(OUTPUT_FILE)
assert len(df_verify) == len(df), "ERROR: Jumlah baris tidak cocok!"
assert 'text_clean' in df_verify.columns, "ERROR: Kolom text_clean tidak ada!"
print(f"\n✅ Verifikasi reload berhasil — {len(df_verify)} baris, kolom text_clean ada.")

---

## 🎉 PIPELINE PHASE 4 — SUMMARY AKHIR

In [ ]:
print("=" * 65)
print(" 🎉  PIPELINE PHASE 4: PREPROCESSING — SELESAI")
print("=" * 65)

checks = {
    "[1] Case folding (lowercase)"              : True,
    "[2] Remove URL"                             : True,
    "[3] Remove mention & hashtag"              : True,
    "[4] Remove digit & punctuation"            : True,
    "[5] Slang normalization (kamus custom)"     : os.path.exists(SLANG_FILE),
    "[6] Tokenization (NLTK)"                   : True,
    "[7] Stopword removal"                       : True,
    "[8] Stemming (Sastrawi)"                   : True,
    "[9] Rare word removal (freq < 2)"          : True,
    "[10] Join tokens → text_clean"             : True,
    "[OUT] dataset_lpdp_preprocessed.csv"        : os.path.exists(OUTPUT_FILE),
    "[OUT] slang_id.csv"                         : os.path.exists(SLANG_FILE),
    "[VAL] text_clean tidak ada NaN"            : df['text_clean'].isna().sum() == 0,
    "[VAL] text_clean tidak ada empty string"   : df['text_clean'].str.strip().eq('').sum() == 0,
}

all_pass = True
for step, status in checks.items():
    icon = '✅' if status else '❌'
    if not status:
        all_pass = False
    print(f"  {icon} {step}")

print()
print(f"  📊 Input  : bertopic_4_topik_final.xlsx → {len(df)} artikel")
print(f"  📊 Output : dataset_lpdp_preprocessed.csv → {len(df)} artikel")
print(f"  📊 Vocab  : {total_vocab:,} → {keep_count:,} kata (hapus {rare_count:,} rare)")
print(f"  📊 Avg token/artikel : {df['token_count'].mean():.1f} token")
print()
print("  -" * 30)
print(f"  📂 Distribusi Sentimen:")
for label, count in df['Sentiment'].value_counts().items():
    print(f"     {label:10s}: {count} ({count/len(df)*100:.1f}%)")
if 'TOPIC_COLUMN' in globals() and TOPIC_COLUMN:
    print(f"  📂 Distribusi Topic Label ({TOPIC_COLUMN}):")
    for label, count in df[TOPIC_COLUMN].value_counts().items():
        print(f"     {str(label):20s}: {count} ({count/len(df)*100:.1f}%)")
else:
    print("  ⚠️ Distribusi Topic Label: kolom topik tidak tersedia")
print()
if all_pass:
    print("  ✅ SEMUA CEK LULUS — dataset_lpdp_preprocessed.csv siap untuk Phase 5!")
else:
    print("  ⚠️  ADA CEK YANG GAGAL — periksa output di atas.")
print("=" * 65)